In [1]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from pydantic import BaseModel, Field
import operator

In [2]:
load_dotenv()

False

In [3]:
model = ChatOpenAI(model='gpt-4o-mini')

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [ ]:
class EvaluationSchema(BaseModel):
    feedback:str = Field(description="Detailed feedback for the essay")
    score :int = Field(description="Score for the essay out of 10",ge=0,le=10)

In [ ]:
structured_model = model.with_structured_output(EvaluationSchema)

In [ ]:
#reducer function add
class UPSCState(TypedDict):
    essay :str
    language_feedback:str
    analysis_feedback:str
    clarity_feedback:str
    overall_feedback:str
    individual_scores:Annotated[list[int],operator.add]
    avg_score:float

In [ ]:
def evaluate_language(state:UPSCState):
    prompt = f"Evaluate the following essay for language quality and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structured_model.predict(prompt)
    return {'language_feedback': result.feedback, 'individual_scores': [result.score]}

In [ ]:
def evaluate_analysis(state:UPSCState):
    prompt = f"Evaluate the following essay for analysis quality and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structured_model.predict(prompt)
    return {'analysis_feedback': result.feedback, 'individual_scores': [result.score]}

In [ ]:
def evaluate_clarity(state:UPSCState):
    prompt = f"Evaluate the following essay for clarity and provide feedback and a score out of 10:\n\n{state['essay']}"
    result = structured_model.predict(prompt)
    return {'clarity_feedback': result.feedback, 'individual_scores': [result.score]}

In [ ]:
def final_evaluation(state:UPSCState):
    avg_score = sum(state['individual_scores']) / len(state['individual_scores'])
    prompt = f"Based on the following feedbacks, provide an overall feedback for the essay:\n\nLanguage Feedback: {state['language_feedback']}\nAnalysis Feedback: {state['analysis_feedback']}\nClarity Feedback: {state['clarity_feedback']}"
    overall_feedback = model.invoke(prompt).content
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [ ]:
graph = StateGraph(UPSCState)

graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_clarity',evaluate_clarity)
graph.add_node('final_evaluation',final_evaluation)

graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_clarity')
graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_clarity','final_evaluation')
graph.add_edge('final_evaluation',END)

workflow = graph.compile()


In [ ]:
initial_state = {
    'essay': "The essay discusses the importance of environmental conservation and the role of individuals in protecting natural resources. It emphasizes the need for sustainable practices and highlights various strategies that can be adopted to reduce environmental impact. The essay also touches upon the consequences of neglecting environmental responsibilities and calls for collective action to address pressing ecological issues.",}

workflow.invoke(initial_state)